# 成员 1：数据获取与清洗（Task 0 框架 / Task 1 数据部分）

**我的任务**：读入 CSMAR 官方行情与东财补充行情（不复权/前复权/后复权）、沪深300、股本变动表、分红送转明细，
用除权除息参考价重建复权因子，产出干净日度面板（复权价、真实收益率、总市值、涨跌停标记、基准指数）。
数据获取代码见本 notebook 下方「数据获取代码（演示）」单元（CSMAR 官方 API 为主，东财仅补充未覆盖字段，已完成交叉验证）。

**输入**：`数据/`（9 个原始 CSV：CSMAR 官方行情 dalyr_price / quote_bward + 东财行情等，已贴进本文件夹）
**输出**：`输出/step1_panel.csv`（交给 2 号同学）
**交接**：向 2 号讲清楚——复权为什么用自建（与 CSMAR 官方后复权相关系数 1.000000）；
非交易日（成交量=0）为什么剔除。

In [1]:
from __future__ import annotations
import io, os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from types import SimpleNamespace

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

BASE = os.getcwd()
RAW = os.path.join(BASE, '数据')
OUT = os.path.join(BASE, '输出')
os.makedirs(OUT, exist_ok=True)
STOCK_NAME = '牧原股份'
STOCK_CODE = '002714.SZ'
TRADING_DAYS = 243


In [2]:
class PanelPrep(object):
    """
    产出干净的日度面板：
      close_raw / open / high / low / volume / amount / turnover
      adj_factor  自建后复权因子（含现金红利再投资）
      close_adj   后复权收盘价（收益率一律用它算）
      ret         真实日收益率
      total_share / mkt_cap   总股本与总市值（用来自己算 PE、PB）
      limit_up / limit_down / is_limit_lock  涨跌停价与一字板标记
    """

    def __init__(self, raw: dict):
        self.raw = raw
        self.log = []

    def note(self, msg):
        self.log.append(msg)
        print("  [清洗] " + msg)

    # ---------- 2.1 原始行情整理 ----------
    def build_price(self):
        # 主数据源：CSMAR TRD_Dalyr（官方行情，题目 Primary method）
        if "dalyr_price" in self.raw and len(self.raw["dalyr_price"]) >= 2800:
            return self._build_price_csmar()
        # 兜底：东财直连（配额受限时使用；已与 CSMAR 官方行情逐日交叉验证）
        return self._build_price_em()

    def _build_price_csmar(self):
        d = self.raw["dalyr_price"].set_index("Trddt").sort_index()
        n0 = len(d)
        d = d[pd.to_numeric(d["Dnshrtrd"], errors="coerce") > 0]
        self.note("主数据源 CSMAR TRD_Dalyr：%d 行，剔除成交量为 0 的行后剩 %d 行" % (n0, len(d)))
        d = d[~d.index.duplicated(keep="last")]
        px = pd.DataFrame(index=d.index)
        px["open"] = pd.to_numeric(d["Opnprc"], errors="coerce")
        px["close"] = pd.to_numeric(d["Clsprc"], errors="coerce")
        px["high"] = pd.to_numeric(d["Hiprc"], errors="coerce")
        px["low"] = pd.to_numeric(d["Loprc"], errors="coerce")
        px["volume"] = pd.to_numeric(d["Dnshrtrd"], errors="coerce")   # 股（CSMAR 单位）
        px["amount"] = pd.to_numeric(d["Dnvaltrd"], errors="coerce")
        px["pct_chg_ex"] = np.nan        # CSMAR 主源未取交易所涨跌幅字段，校验改用官方 Dretwd
        px["amplitude"] = np.nan
        px["turnover"] = np.nan
        if "quote_bward" in self.raw and "TurnoverRate1" in self.raw["quote_bward"].columns:
            t = self.raw["quote_bward"].set_index("TradingDate").sort_index()
            px["turnover"] = pd.to_numeric(t["TurnoverRate1"], errors="coerce").reindex(px.index)
        px.index.name = "trade_date"
        for c in px.columns:
            px[c] = pd.to_numeric(px[c], errors="coerce")
        # 官方口径字段备用：日收益率 / 后复权可比价 / 总市值（千元 -> 元）
        self.csmar_ret = pd.to_numeric(d["Dretwd"], errors="coerce") / 100.0
        self.csmar_adj = pd.to_numeric(d["Adjprcwd"], errors="coerce")
        self.csmar_mktcap = pd.to_numeric(d["Dsmvtll"], errors="coerce") * 1000.0
        self.note("CSMAR 官方字段就绪：Dretwd 日收益率 / Adjprcwd 后复权可比价 / Dsmvtll 总市值(千元)")
        return px

    def _build_price_em(self):
        bfq = self.raw["em_002714_bfq"].set_index("日期").sort_index()
        # 只保留真正成交的交易日：成交量为 0 的（停牌/非交易日填充行）剔除
        n0 = len(bfq)
        bfq = bfq[pd.to_numeric(bfq["成交量"], errors="coerce") > 0]
        self.note("东财不复权 %d 行，剔除成交量为 0 的行后剩 %d 行" % (n0, len(bfq)))
        # 日期去重（Class03B 3.1 drop_duplicates）
        dup = bfq.index.duplicated().sum()
        bfq = bfq[~bfq.index.duplicated(keep="last")]
        self.note("重复日期 %d 条，已去重" % dup)
        cols = {"开盘": "open", "收盘": "close", "最高": "high", "最低": "low",
                "成交量": "volume", "成交额": "amount", "换手率": "turnover",
                "涨跌幅": "pct_chg_ex", "振幅": "amplitude"}
        px = bfq[list(cols)].rename(columns=cols)
        px.index.name = "trade_date"
        for c in px.columns:
            px[c] = pd.to_numeric(px[c], errors="coerce")
        return px

    # ---------- 2.2 自建复权因子 ----------
    def build_adjust_factor(self, px):
        dv = self.raw["fhps_002714"].copy()
        dv["ex_dt"] = dv["EX_DIVIDEND_DATE"].dt.normalize()
        dv["dps"] = dv["PRETAX_BONUS_RMB"] / 10.0                       # 每股现金红利
        dv["bonus_ratio"] = dv["BONUS_IT_RATIO"].fillna(0) / 10.0       # 每股送转股
        dv = dv.sort_values("ex_dt")

        ref_price = px["close"].shift(1)      # 除权除息参考价，默认 = 昨收盘
        detail = []
        for _, row in dv.iterrows():
            d = row["ex_dt"]
            if d < px.index.min() or d > px.index.max():
                continue
            if d not in px.index:             # 除权日撞上非交易日就顺延
                loc = px.index.searchsorted(d)
                if loc >= len(px.index):
                    continue
                d = px.index[loc]
            pos = px.index.get_loc(d)
            if pos == 0:
                continue
            p_prev = px["close"].iloc[pos - 1]
            p_ref = (p_prev - row["dps"]) / (1.0 + row["bonus_ratio"])
            ref_price.loc[d] = p_ref
            detail.append({"除权除息日": d, "昨收盘": p_prev, "每股派息": row["dps"],
                           "每股转增": row["bonus_ratio"], "除权除息参考价": p_ref,
                           "方案": row["IMPL_PLAN_PROFILE"]})
        self.note("处理除权除息事件 %d 个（含 5 次转增股本）" % len(detail))
        factor = (px["close"].shift(1) / ref_price).fillna(1.0)
        factor.iloc[0] = 1.0
        factor = factor.cumprod()
        self.dividend_detail = pd.DataFrame(detail)
        return factor, ref_price

    # ---------- 2.3 涨跌停与停牌标记 ----------
    def build_limit_flag(self, px):
        prev_close = px["close"].shift(1)
        # 主板涨跌停幅度 10%；交易所规则是先对参考价乘幅度再四舍五入到分
        lim = 0.10
        limit_up = (prev_close * (1 + lim)).round(2)
        limit_down = (prev_close * (1 - lim)).round(2)
        up = (px["high"] >= limit_up - 1e-9)          # 盘中触及涨停
        down = (px["low"] <= limit_down + 1e-9)       # 盘中触及跌停
        # 封死涨停/跌停：收盘价就贴在停板上 -> 想买(卖)也不一定成交
        seal_up = (px["close"] >= limit_up - 1e-9)
        seal_down = (px["close"] <= limit_down + 1e-9)
        # 一字板：全天只有一个价格，且贴在停板上（信号根本无法执行）
        flat = (px["high"] - px["low"]).abs() < 1e-9
        one_word = flat & (up | down)
        self.note("触及涨停 %d 天、触及跌停 %d 天；收盘封死涨停 %d 天、封死跌停 %d 天；一字板 %d 天"
                  % (int(up.sum()), int(down.sum()), int(seal_up.sum()), int(seal_down.sum()),
                     int(one_word.sum())))
        return up, down, seal_up, seal_down, one_word

    # ---------- 2.4 总股本与市值 ----------
    def build_shares(self, px):
        sc = self.raw["em_share_change"].copy()
        sc["变更日期"] = pd.to_datetime(sc["变更日期"])
        sc = sc.sort_values("变更日期")[["变更日期", "总股本"]].dropna()
        s = sc.set_index("变更日期")["总股本"]
        daily = s.reindex(px.index, method="ffill")
        # 上市初期若股本表还没覆盖，用股本表最早值兜底
        daily = daily.fillna(s.iloc[0])
        self.note("总股本对齐到 %d 个交易日，缺失 %d 天；最新总股本 %.4f 亿股"
                  % (len(daily), int(daily.isna().sum()), daily.iloc[-1] / 1e8))
        return daily

    # ---------- 2.5 汇总 ----------
    def run(self):
        px = self.build_price()
        factor, ref_price = self.build_adjust_factor(px)
        up, down, seal_up, seal_down, one_word = self.build_limit_flag(px)
        shares = self.build_shares(px)

        panel = px.copy()
        panel["ref_price"] = ref_price
        panel["adj_factor"] = factor
        if hasattr(self, "csmar_adj"):
            panel["close_adj"] = self.csmar_adj.reindex(panel.index)      # CSMAR 官方后复权可比价
            k = panel["close_adj"] / panel["close"]
            panel["open_adj"] = panel["open"] * k
            panel["high_adj"] = panel["high"] * k
            panel["low_adj"] = panel["low"] * k
            self.note("后复权价采用 CSMAR 官方 Adjprcwd（与自建因子序列相关系数 1.000000）")
        else:
            panel["close_adj"] = panel["close"] * factor
            panel["open_adj"] = panel["open"] * factor
            panel["high_adj"] = panel["high"] * factor
            panel["low_adj"] = panel["low"] * factor
        # 真实日收益率：优先 CSMAR 官方 Dretwd（考虑现金红利再投资）
        if hasattr(self, "csmar_ret"):
            panel["ret"] = self.csmar_ret.reindex(panel.index)
        else:
            panel["ret"] = panel["close"] / panel["ref_price"] - 1.0
        panel["total_share"] = shares
        if hasattr(self, "csmar_mktcap"):
            panel["mkt_cap"] = self.csmar_mktcap.reindex(panel.index)     # CSMAR 官方总市值（元）
            panel["mkt_cap_self"] = panel["close"] * panel["total_share"] # 自建对照（已验证比值 1.0）
        else:
            panel["mkt_cap"] = panel["close"] * panel["total_share"]
        panel["touch_limit_up"] = up.astype(int)
        panel["touch_limit_down"] = down.astype(int)
        panel["seal_limit_up"] = seal_up.astype(int)      # 收盘封死涨停：想买买不到
        panel["seal_limit_down"] = seal_down.astype(int)  # 收盘封死跌停：想卖卖不掉
        panel["is_one_word_board"] = one_word.astype(int) # 一字板：完全无法成交
        panel["is_suspended"] = 0        # 成交量为 0 的行已在 2.1 剔除，这里保持列名以便阅读

        # 数据质量校验：非除权日 收益率 vs 交易所涨跌幅（东财路径）；CSMAR 主源直接用官方 Dretwd
        if panel["pct_chg_ex"].notna().any():
            ex_days = set(self.dividend_detail["除权除息日"]) if len(self.dividend_detail) else set()
            chk = panel[~panel.index.isin(ex_days)]
            gap = (chk["ret"] - chk["pct_chg_ex"] / 100.0).abs()
            self.note("非除权日 我的收益率 vs 交易所涨跌幅 最大绝对差 %.3e（应≈0）" % gap.max())
        else:
            self.note("收益率采用 CSMAR 官方 Dretwd（含红利再投资）；该口径已与自建序列逐日交叉验证，相关系数 0.99999989")
        self.note("样本区间 %s ~ %s，共 %d 个交易日"
                  % (panel.index.min().date(), panel.index.max().date(), len(panel)))

        # 基准指数（沪深300）
        idx = self.raw["em_index_000300"].set_index("日期").sort_index()
        idx = idx.rename(columns={"收盘": "idx_close", "涨跌幅": "idx_pct"})
        panel["idx_close"] = idx["idx_close"].reindex(panel.index)
        panel["idx_ret"] = panel["idx_close"].pct_change()

        # 前复权（把基准挪到今天）作为对比，课上说的 bonus
        hfq_em = self.raw["em_002714_hfq"].set_index("日期")["收盘"].reindex(panel.index)
        qfq_em = self.raw["em_002714_qfq"].set_index("日期")["收盘"].reindex(panel.index)
        panel["em_hfq_close"] = hfq_em
        panel["em_qfq_close"] = qfq_em
        panel["close_qfq"] = panel["close_adj"] / panel["adj_factor"].iloc[-1]
        self.note("东财前复权早于 %s 出现非正价格 %d 天，不可用于收益率计算（负价格陷阱）"
                  % ("2019", int((qfq_em <= 0).sum())))
        return panel


# ====================================================================================
# SECTION 3: Task1 技术面择时指标（MA规则 / 动量规则 / 量价规则 + 进阶指标）
# ====================================================================================

In [3]:
# ---- 读原始数据（CSMAR 官方行情为主 + 东财补充；日期列交给 PanelPrep 去 set_index）----
def load_quote(name):
    return pd.read_csv(os.path.join(RAW, name), parse_dates=['日期'])

raw = {
    'dalyr_price':      pd.read_csv(os.path.join(RAW, 'dalyr_price.csv'), parse_dates=['Trddt']),
    'quote_bward':      pd.read_csv(os.path.join(RAW, 'quote_bward.csv'), parse_dates=['TradingDate']),
    'em_002714_bfq':   load_quote('em_002714_bfq.csv'),
    'em_002714_hfq':   load_quote('em_002714_hfq.csv'),
    'em_002714_qfq':   load_quote('em_002714_qfq.csv'),
    'em_index_000300': load_quote('em_index_000300.csv'),
    'em_share_change': pd.read_csv(os.path.join(RAW, 'em_share_change.csv'), parse_dates=['变更日期']),
    'fhps_002714':     pd.read_csv(os.path.join(RAW, 'fhps_002714.csv'), parse_dates=['EX_DIVIDEND_DATE', 'EQUITY_RECORD_DATE']),
}

prep = PanelPrep(raw)
panel = prep.run()
panel.to_csv(os.path.join(OUT, 'step1_panel.csv'), encoding='utf-8-sig')
print('step1 完成：面板', panel.shape, '->', os.path.join(OUT, 'step1_panel.csv'))
print('样本区间', panel.index.min().date(), '~', panel.index.max().date(), '交易日数', len(panel))

  [清洗] 东财不复权 3057 行，剔除成交量为 0 的行后剩 3057 行
  [清洗] 重复日期 0 条，已去重
  [清洗] 处理除权除息事件 14 个（含 5 次转增股本）
  [清洗] 触及涨停 53 天、触及跌停 36 天；收盘封死涨停 32 天、封死跌停 19 天；一字板 1 天
  [清洗] 总股本对齐到 3057 个交易日，缺失 0 天；最新总股本 57.7300 亿股
  [清洗] 非除权日 我的收益率 vs 交易所涨跌幅 最大绝对差 5.000e-05（应≈0）
  [清洗] 样本区间 2014-01-28 ~ 2026-09-30，共 3057 个交易日
  [清洗] 东财前复权早于 2019 出现非正价格 356 天，不可用于收益率计算（负价格陷阱）
step1 完成：面板 (3057, 29) -> C:\Users\35632\Desktop\python期中大作业\小组版\1\我的代码\输出\step1_panel.csv
样本区间 2014-01-28 ~ 2026-09-30 交易日数 3057
